# 合成関数と連鎖律（チェインルール）

このノートブックでは、ニューラルネットワークのパラメータ最適化に必要な**合成関数の微分（連鎖律）**と**偏微分**について学びます。

## 目次
1. 合成関数とは（入れ子構造）
2. 連鎖律（チェインルール）の基礎
3. ニューラルネットワークにおける入れ子構造
4. 偏微分による損失関数の最適化
5. 重みパラメータの更新式の導出

## 対応する教科書のセクション
- 3-9: 出力層のパラメータの影響範囲を考察する ー合成関数ー
- 3-10: 出力層のパラメータによって損失関数を最適化する ー偏微分ー

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle

## 1. 合成関数とは（入れ子構造）（図3.26）

ニューラルネットワークは、複数の関数が**入れ子**になった構造をしています。

### 基礎となる数理的手法と情報工学的アプローチ

| 数学概念 | 記号・定義 |
|---------|----------|
| **合成関数の微分（連鎖律）** | $\{f(g(x))\}' = f'(g(x)) \cdot g'(x)$ |
| **偏導関数** | $\frac{\partial}{\partial x}f(x,y) = \lim_{h \to 0} \frac{f(x+h,y) - f(x,y)}{h}$ |
| **分数関数の微分** | $\left(\frac{f(x)}{g(x)}\right)' = \frac{f'(x)g(x) - f(x)g'(x)}{(g(x))^2}$ |

| 情報工学概念 | 記号・定義 |
|------------|----------|
| **対数尤度関数を用いた損失関数** | $L(P) = -\sum_{k=1}^{n} t_k \log P_k$ |
| **ソフトマックス関数** | $\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{k=1}^{n} e^{x_k}}$ |

In [ ]:
# 図3.26: 基礎となる数理的手法と情報工学的アプローチ

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)

# タイトル
ax.text(7, 9.5, '図3.26: 本節以降で解説する数理的手法及び情報工学的なアプローチ', 
        fontsize=13, fontweight='bold', ha='center')

# 左側: 数理的手法
box1 = FancyBboxPatch((0.5, 4), 6, 5, boxstyle='round,pad=0.1', 
                       facecolor='lightblue', edgecolor='blue', linewidth=2)
ax.add_patch(box1)
ax.text(3.5, 8.5, '基礎となる数理的手法', fontsize=12, fontweight='bold', ha='center')

ax.text(1, 7.5, '合成関数の微分（連鎖律）:', fontsize=10, fontweight='bold')
ax.text(1, 6.8, r"$\{f(g(x))\}' = f'(g(x)) \cdot g'(x)$", fontsize=11)

ax.text(1, 6, '偏導関数:', fontsize=10, fontweight='bold')
ax.text(1, 5.3, r'$\frac{\partial}{\partial x}f(x,y) = \lim_{h \to 0} \frac{f(x+h,y) - f(x,y)}{h}$', fontsize=10)

ax.text(1, 4.5, '分数関数の微分:', fontsize=10, fontweight='bold')

# 右側: 情報工学的アプローチ
box2 = FancyBboxPatch((7.5, 4), 6, 5, boxstyle='round,pad=0.1',
                       facecolor='lightyellow', edgecolor='orange', linewidth=2)
ax.add_patch(box2)
ax.text(10.5, 8.5, '情報工学的なアプローチ', fontsize=12, fontweight='bold', ha='center')

ax.text(8, 7.5, '対数尤度関数を用いた損失関数の定義:', fontsize=10, fontweight='bold')
ax.text(8, 6.8, r'$L(P) = -\sum_{k=1}^{n} t_k \log P_k$', fontsize=11)

ax.text(8, 6, 'ソフトマックス関数:', fontsize=10, fontweight='bold')
ax.text(8, 5.3, r'$\mathrm{softmax}(x_i) = \frac{e^{x_i}}{\sum_{k=1}^{n} e^{x_k}}$', fontsize=11)

ax.text(8, 4.5, '重みパラメータの更新式:', fontsize=10, fontweight='bold')

# 下部: 更新式のボックス
box3 = FancyBboxPatch((2, 0.5), 10, 3, boxstyle='round,pad=0.1',
                       facecolor='lightgreen', edgecolor='green', linewidth=2)
ax.add_patch(box3)
ax.text(7, 3, '重みパラメータの更新式', fontsize=12, fontweight='bold', ha='center')

ax.text(2.5, 2.2, r'$w_{1,0}^{fc(n)} = w_{1,0}^{fc(n-1)} - \eta \cdot \frac{\partial L(P)}{\partial w_{1,0}^{fc(n-1)}}$', fontsize=11)
ax.text(2.5, 1.5, r'$\frac{\partial L(P)}{\partial w_{1,0}^{fc}} = (P_0 - t_0) z_1$', fontsize=11)

ax.text(7.5, 2.2, r'$n$: 自然数（更新回数）', fontsize=10)
ax.text(7.5, 1.6, r'$w_{1,0}^{fc(n)}$: $n$回更新後の重みパラメータ', fontsize=10)
ax.text(7.5, 1.0, r'$\eta$: 学習率', fontsize=10)

ax.axis('off')
plt.tight_layout()
plt.show()

## 2. 連鎖律（チェインルール）の基礎

### 2.1 合成関数の微分公式

$$\{f(g(x))\}' = f'(g(x)) \cdot g'(x)$$

$g(x) = u$, $f(u) = y$ と表記すると、この公式は次のように書き換えられます：

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

連鎖律の計算プロセスを示す際はこのような形式で表記されることが多いように見受けられ、本書でもこの形式を主に採用して解説を進めます。

### 2.2 偏微分への適用

この合成関数の微分（連鎖律）は偏微分の計算に適用することがしばしばあり、その際の表記は次の通りです：

$$\frac{\partial y}{\partial x} = \frac{\partial y}{\partial u} \cdot \frac{\partial u}{\partial x}$$

### Lesson: 合成関数の微分

公式の導出方法は割愛しますが、合成関数の微分（連鎖律）はAIの領域でさまざまなパターンで登場しますので、とにかく慣れることが大切です。本書でも以降で頻出しますので、準備として2つの例題について考えてみましょう。

#### 例題1: $f(x) = (3x^2 + 4)^2$ を $x$ について微分してみましょう

$g(x) = 3x^2 + 4$ として、$g(x) = u$ と置くと $y = f(u) = u^2$ と表記できるので、計算結果は次のようになります：

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = \frac{d(u^2)}{du} \cdot \frac{d(3x^2+4)}{dx}$$

$$= 2 \times u^{2-1} \times 2 \times 3x^{2-1} = 12xu = 12x(3x^2 + 4) = 36x^3 + 48x$$

#### 例題2: $f(x) = \sqrt{3x^2 + 4}$ を $x$ について微分してみましょう

$g(x) = 3x^2 + 4$ として、$g(x) = u$ と置くと $y = f(u) = \sqrt{u} = u^{\frac{1}{2}}$ と表記できるので、計算結果は次の通りとなります：

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = \frac{d(u^{\frac{1}{2}})}{du} \cdot \frac{d(3x^2+4)}{dx}$$

$$= \frac{1}{2} \times u^{\frac{1}{2}-1} \times 2 \times 3x^{2-1} = 3xu^{-\frac{1}{2}} = \frac{3x}{\sqrt{3x^2+4}}$$

本書では合成関数の微分（連鎖律）の計算パターンが随所に現れますので、ぜひ習熟してください。

In [ ]:
# 連鎖律の例を数値的に確認

def numerical_derivative(f, x, h=1e-7):
    """数値微分を計算"""
    return (f(x + h) - f(x - h)) / (2 * h)

# 例題1: f(x) = (3x² + 4)²
def f1(x):
    return (3*x**2 + 4)**2

def f1_derivative_analytical(x):
    """解析的な導関数: 36x³ + 48x"""
    return 36*x**3 + 48*x

# 例題2: f(x) = √(3x² + 4)
def f2(x):
    return np.sqrt(3*x**2 + 4)

def f2_derivative_analytical(x):
    """解析的な導関数: 3x / √(3x² + 4)"""
    return 3*x / np.sqrt(3*x**2 + 4)

print("=== 連鎖律の確認 ===")
print()
print("例題1: f(x) = (3x² + 4)²")
print("解析的な導関数: f'(x) = 36x³ + 48x")
print()

for x in [1, 2, 3]:
    numerical = numerical_derivative(f1, x)
    analytical = f1_derivative_analytical(x)
    print(f"  x = {x}: 数値微分 = {numerical:.4f}, 解析解 = {analytical:.4f}")

print()
print("例題2: f(x) = √(3x² + 4)")
print("解析的な導関数: f'(x) = 3x / √(3x² + 4)")
print()

for x in [1, 2, 3]:
    numerical = numerical_derivative(f2, x)
    analytical = f2_derivative_analytical(x)
    print(f"  x = {x}: 数値微分 = {numerical:.4f}, 解析解 = {analytical:.4f}")

In [ ]:
# 連鎖律の可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(-2, 2, 100)

# 例題1のグラフ
ax = axes[0]
ax.plot(x, f1(x), 'b-', linewidth=2, label=r'$f(x) = (3x^2+4)^2$')
ax.plot(x, f1_derivative_analytical(x), 'r--', linewidth=2, label=r"$f'(x) = 36x^3+48x$")
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(r'例題1: $f(x) = (3x^2+4)^2$ の微分', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# 例題2のグラフ
ax = axes[1]
ax.plot(x, f2(x), 'b-', linewidth=2, label=r'$f(x) = \sqrt{3x^2+4}$')
ax.plot(x, f2_derivative_analytical(x), 'r--', linewidth=2, label=r"$f'(x) = \frac{3x}{\sqrt{3x^2+4}}$")
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(r'例題2: $f(x) = \sqrt{3x^2+4}$ の微分', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. ニューラルネットワークにおける入れ子構造（図3.27, 3.28）

では、損失関数 $L$ を最適化するために勾配降下法によって各パラメータを最適化しましょう。まずは出力層に近いパラメータ $w_{1,0}^{fc}$ を最適化することを考えます。

すると、正解値を起点に考えていくと、図3.27の通り $w_{1,0}^{fc}$ は青で囲った要素に影響を与えていることがわかります。この「**パラメータの影響範囲**」が以降の解説で欠かせない視点となります。

### 3.1 ネットワーク構造（図3.27）

- **入力層**: $x_{i,j}$ (3×3の入力)
- **畳み込み層**: 重み $w_{i,j}^l$ によるフィルタ演算
- **プーリング層**: $z_1, z_2$（特徴量の圧縮）
- **全結合層**: 重み $w_{i,j}^{fc}$ による線形変換
- **出力層**: $P_0, P_1, P_2$（ソフトマックス出力）
- **正解値**: $t_0, t_1, t_2$

In [ ]:
# 図3.27: ネットワーク構造と全結合層のパラメータ

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 12)

# タイトル
ax.text(9, 11.5, '図3.27: 更新の対象となる全結合層のパラメータ及びその影響を受ける関数', 
        fontsize=13, fontweight='bold', ha='center')

# 入力層 (3x3)
ax.text(1.5, 10, '入力層', fontsize=10, fontweight='bold', ha='center')
for i in range(3):
    for j in range(3):
        rect = plt.Rectangle((0.5 + j*0.7, 8.5 - i*0.7), 0.6, 0.6, 
                             facecolor='lightgray', edgecolor='black')
        ax.add_patch(rect)
        ax.text(0.8 + j*0.7, 8.8 - i*0.7, f'$x_{{{i+1},{j+1}}}$', fontsize=7, ha='center', va='center')

# 畳み込み層 (重み)
ax.text(4, 10, '畳み込み層\n(ストライド幅:1)', fontsize=10, fontweight='bold', ha='center')
for k in range(2):  # 2つのフィルタ
    for i in range(2):
        for j in range(2):
            rect = plt.Rectangle((3 + j*0.6, 8.2 - i*0.6 - k*1.5), 0.5, 0.5,
                                 facecolor='lightblue', edgecolor='black')
            ax.add_patch(rect)
            ax.text(3.25 + j*0.6, 8.45 - i*0.6 - k*1.5, f'$w_{{{i+1},{j+1}}}^{k+1}$', 
                   fontsize=6, ha='center', va='center')

# 畳み込み出力 (c)
for k in range(2):
    for i in range(2):
        for j in range(2):
            rect = plt.Rectangle((5 + j*0.6, 8.2 - i*0.6 - k*1.5), 0.5, 0.5,
                                 facecolor='lightyellow', edgecolor='black')
            ax.add_patch(rect)
            ax.text(5.25 + j*0.6, 8.45 - i*0.6 - k*1.5, f'$c_{{{i+1},{j+1}}}^{k+1}$',
                   fontsize=6, ha='center', va='center')

# プーリング層
ax.text(7.5, 10, 'プーリング層', fontsize=10, fontweight='bold', ha='center')
circle1 = Circle((7.5, 8.5), 0.4, facecolor='lightgreen', edgecolor='black')
circle2 = Circle((7.5, 6.5), 0.4, facecolor='lightgreen', edgecolor='black')
ax.add_patch(circle1)
ax.add_patch(circle2)
ax.text(7.5, 8.5, '$z_1$', fontsize=10, ha='center', va='center')
ax.text(7.5, 6.5, '$z_2$', fontsize=10, ha='center', va='center')

# 全結合層
ax.text(10.5, 10, '全結合層', fontsize=10, fontweight='bold', ha='center')
fc_neurons = [(10.5, 8.5, 'fc_0'), (10.5, 7.5, 'fc_1'), (10.5, 6.5, 'fc_2')]
for x, y, label in fc_neurons:
    circle = Circle((x, y), 0.35, facecolor='orange', edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, f'${label}$', fontsize=9, ha='center', va='center')

# 重み線（全結合層）- ハイライト
ax.annotate('', xy=(10.15, 8.5), xytext=(7.9, 8.5),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(8.5, 8.9, '$w_{1,0}^{fc}$', fontsize=10, color='blue', fontweight='bold')

# その他の重み線
other_weights = [
    (7.9, 6.5, 10.15, 8.5), (7.9, 8.5, 10.15, 7.5), (7.9, 6.5, 10.15, 7.5),
    (7.9, 8.5, 10.15, 6.5), (7.9, 6.5, 10.15, 6.5),
]
for x1, y1, x2, y2 in other_weights:
    ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.5, linewidth=0.8)

# 出力層
ax.text(13, 10, '出力層', fontsize=10, fontweight='bold', ha='center')
output_neurons = [(13, 8.5, '$P_0$'), (13, 7.5, '$P_1$'), (13, 6.5, '$P_2$')]
for x, y, label in output_neurons:
    circle = Circle((x, y), 0.35, facecolor='coral', edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, label, fontsize=10, ha='center', va='center')

# 全結合層から出力層への線
for fc_x, fc_y, _ in fc_neurons:
    for out_x, out_y, _ in output_neurons:
        ax.plot([fc_x + 0.35, out_x - 0.35], [fc_y, out_y], 'gray', alpha=0.3, linewidth=0.5)

# 正解値
ax.text(15, 10, '正解値', fontsize=10, fontweight='bold', ha='center')
targets = [(15, 8.5, '$t_0$'), (15, 7.5, '$t_1$'), (15, 6.5, '$t_2$')]
for x, y, label in targets:
    circle = Circle((x, y), 0.35, facecolor='white', edgecolor='black', linestyle='--')
    ax.add_patch(circle)
    ax.text(x, y, label, fontsize=10, ha='center', va='center')

# 出力と正解値の接続（比較）
for i in range(3):
    ax.annotate('', xy=(14.65, 8.5 - i), xytext=(13.35, 8.5 - i),
                arrowprops=dict(arrowstyle='<->', color='red', lw=1))

# パラメータの影響範囲を示すボックス
highlight_box = FancyBboxPatch((9.8, 6), 3.7, 3.2, boxstyle='round,pad=0.1',
                                facecolor='none', edgecolor='blue', linestyle='--', linewidth=2)
ax.add_patch(highlight_box)
ax.text(11.5, 5.5, '$w_{1,0}^{fc}$ が影響を与える範囲', fontsize=10, color='blue', ha='center')

# 下部に説明
ax.text(9, 4, '青でハイライトされている $w_{1,0}^{fc}$ は青で囲った要素に影響を与えています。', 
        fontsize=11, ha='center')
ax.text(9, 3.3, 'この「パラメータの影響範囲」が以降の解説で欠かせない視点となります。', 
        fontsize=11, ha='center', color='blue')

ax.axis('off')
plt.tight_layout()
plt.show()

### 3.2 関数の入れ子構造（図3.28）

ここで条件を再確認しましょう。損失関数 $L(P)$ は3つのソフトマックス関数 $P_0 = s_0$, $P_1 = s_1$, $P_2 = s_2$ を含んでいます。そして、それぞれのソフトマックス関数 $s_0, s_1, s_2$ は全結合層の $fc_0, fc_1, fc_2$ の影響を受けています。さらに、この $fc_0, fc_1, fc_2$ も $w^{fc}$ を重みパラメータとする関数となっています。この点をしっかり確認できるように各数理モデルを再掲します。

$$L(P) = -(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2) \tag{3-12}$$

$$P_0 = s_0(fc_0, fc_1, fc_2) = \frac{e^{fc_0(z_1,z_2)}}{\sum_{k=0}^{2} e^{fc_k(z_1,z_2)}} \tag{3-13}$$

$$P_1 = s_1(fc_0, fc_1, fc_2) = \frac{e^{fc_1(z_1,z_2)}}{\sum_{k=0}^{2} e^{fc_k(z_1,z_2)}}$$

$$P_2 = s_2(fc_0, fc_1, fc_2) = \frac{e^{fc_2(z_1,z_2)}}{\sum_{k=0}^{2} e^{fc_k(z_1,z_2)}}$$

$$fc_0 = fc_0(z_1, z_2) = w_{1,0}^{fc} z_1 + w_{2,0}^{fc} z_2 \tag{3-14}$$

これらの関数の中には、同じ要素が繰り返し登場しています。その点を可視化すると、上記の3つの数式は図3.28のように「**入れ子構造**」のように設計されていることがわかります。このメカニズムを理解することが、深層学習モデルのパラメータ更新を理解するためには不可欠です。

In [ ]:
# 図3.28: 入れ子構造の可視化

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)

# タイトル
ax.text(7, 11.5, '図3.28: 詳細 - 更新の対象となる全結合層のパラメータ及びその影響を受ける関数',
        fontsize=12, fontweight='bold', ha='center')

# 外側のボックス: L(P)
box_L = FancyBboxPatch((1, 1), 12, 9.5, boxstyle='round,pad=0.1',
                        facecolor='lightyellow', edgecolor='black', linewidth=2, alpha=0.3)
ax.add_patch(box_L)

# 損失関数
ax.text(1.5, 10, r'$L(P) = -(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)$', 
        fontsize=11, fontweight='bold')

# 中間のボックス: P_0 = s_0
box_P = FancyBboxPatch((2, 3.5), 10, 5.5, boxstyle='round,pad=0.1',
                        facecolor='lightblue', edgecolor='blue', linewidth=2, alpha=0.3)
ax.add_patch(box_P)

# P_0 の定義
ax.text(2.5, 8.5, r'$P_0 = s_0(fc_0, fc_1, fc_2) = \frac{e^{fc_0(z_1,z_2)}}{\sum_{k=0}^{2} e^{fc_k(z_1,z_2)}}$', 
        fontsize=11, color='blue')

# 内側のボックス: fc_0
box_fc = FancyBboxPatch((3, 4), 8, 3.5, boxstyle='round,pad=0.1',
                         facecolor='lightgreen', edgecolor='green', linewidth=2, alpha=0.5)
ax.add_patch(box_fc)

# fc_0 の定義
ax.text(3.5, 6.8, r'$fc_0 = fc_0(z_1, z_2) = w_{1,0}^{fc} z_1 + w_{2,0}^{fc} z_2$', 
        fontsize=11, color='green', fontweight='bold')

# ハイライト: w_{1,0}^{fc}
highlight = FancyBboxPatch((4.5, 4.5), 2.5, 1.5, boxstyle='round,pad=0.1',
                            facecolor='yellow', edgecolor='red', linewidth=2)
ax.add_patch(highlight)
ax.text(5.75, 5.25, r'$w_{1,0}^{fc}$', fontsize=14, fontweight='bold', ha='center', va='center', color='red')

# 入れ子構造の説明
ax.text(7, 2.5, r'$L(P) = -(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)$ の中に、', fontsize=10)
ax.text(7, 2, r'$P_0 = s_0(fc_0, fc_1, fc_2)$ が含まれている', fontsize=10, color='blue')
ax.text(7, 1.3, r'$P_0 = s_0(fc_0, fc_1, fc_2)$ の中に、', fontsize=10)
ax.text(7, 0.8, r'$fc_0 = fc_0(z_1, z_2)$ が含まれている', fontsize=10, color='green')

# 矢印で入れ子を示す
ax.annotate('', xy=(2.3, 8.2), xytext=(2.3, 9.5),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.annotate('', xy=(3.3, 6.5), xytext=(3.3, 8),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.axis('off')
plt.tight_layout()
plt.show()

print("この入れ子構造は、関数の中にさらに関数を包含するという構造になっています。")
print("このように関数の中に関数が組み込まれた関数を【合成関数】と言います。")
print()
print("図中で青くハイライトしているパラメータ w_{1,0}^{fc} の最適化を考えると、")
print("w_{1,0}^{fc} は fc_0 から P_0、さらに P_0 から L(P) へと順に影響しているため、")
print("1つのパラメータが複数の関数に影響している様子がうかがえます。")
print()
print("そのため、複数の関数への微分の影響を考慮した【合成関数の微分】と呼ばれる")
print("操作が必要となります。この手法は【連鎖律】とも呼ばれます。")

## 4. 偏微分による損失関数の最適化（3-10節）

では、順番に連鎖律を適用していきましょう。$L(P)$ を $w_{1,0}^{fc}$ で偏微分する際、$w_{1,0}^{fc}$ は $fc_0$ を経由して $L(P)$ に影響しています。そこで、まずは式(3-15)に示す偏微分操作を行います。

### 4.1 連鎖律の適用

$$\frac{\partial L(P)}{\partial w_{1,0}^{fc}} = \frac{\partial L(P)}{\partial fc_0} \cdot \frac{\partial fc_0}{\partial w_{1,0}^{fc}} \tag{3-15}$$

この式(3-15)をよく見ると、$L(P)$ を $fc_0$ で偏微分したものと、$fc_0$ を $w_{1,0}^{fc}$ で偏微分した計算結果との**積**になっています。このような形になるのが連鎖律の特徴です。

### 4.2 $\frac{\partial fc_0}{\partial w_{1,0}^{fc}}$ の計算

すると、右辺の積の右側 $\frac{\partial fc_0}{\partial w_{1,0}^{fc}}$ についてよく考えてみると式(3-14)の $fc_0 = w_{1,0}^{fc} z_1 + w_{2,0}^{fc} z_2$ より、$fc_0$ を $w_{1,0}^{fc}$ で微分計算すると式(3-16)の通りです。

$$\frac{\partial L(P)}{\partial w_{1,0}^{fc}} = \frac{\partial L(P)}{\partial fc_0} \cdot \frac{\partial}{\partial w_{1,0}^{fc}}(w_{1,0}^{fc} z_1 + w_{2,0}^{fc} z_2) = \frac{\partial L(P)}{\partial fc_0} \cdot z_1 \tag{3-16}$$

細かいですが $w_{2,0}^{fc} z_2$ は $w_{1,0}^{fc}$ で微分すると0になります。その理由は第2章の「Lesson：微分・偏微分の計算例」の解説を踏まえれば理解できるはずですので、わからなければ読み返して確認しましょう。

In [ ]:
# 偏微分の計算例

print("=== fc_0 を w_{1,0}^{fc} で偏微分 ===")
print()
print("fc_0 = w_{1,0}^{fc} * z_1 + w_{2,0}^{fc} * z_2")
print()
print("∂fc_0/∂w_{1,0}^{fc} = ∂/∂w_{1,0}^{fc} (w_{1,0}^{fc} * z_1 + w_{2,0}^{fc} * z_2)")
print("                    = z_1 + 0")
print("                    = z_1")
print()
print("※ w_{2,0}^{fc} * z_2 は w_{1,0}^{fc} を含まないので、微分すると0になります")

### 4.3 $\frac{\partial L(P)}{\partial fc_0}$ の計算

次に式(3-15)の $\frac{\partial L(P)}{\partial fc_0}$ について考えます。$fc_0$ は $P_0$, $P_1$, $P_2$ を経由して式(3-12)に影響しています。その様子は次の数式を眺めれば一目瞭然です。

$$P_0 = s_0(fc_0, fc_1, fc_2) = \frac{e^{fc_0}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$$

$$P_1 = s_1(fc_0, fc_1, fc_2) = \frac{e^{fc_1}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$$

$$P_2 = s_2(fc_0, fc_1, fc_2) = \frac{e^{fc_2}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$$

つまり、$fc_0$ は $P_0$, $P_1$, $P_2$ の**すべてに影響**を与えているということです。

よって、$\frac{\partial L(P)}{\partial fc_0}$ の形に従っていきなり損失関数 $L(P)$ を偏微分するのではなく、まずは $P_0$, $P_1$, $P_2$ それぞれを $fc_0$ で偏微分するというステップが不可欠となります。よって、$\frac{\partial L(P)}{\partial fc_0}$ を計算するには式(3-17)のように $P_0$, $P_1$, $P_2$ それぞれにおける偏微分結果を足し合わせる処理が行われます。

$$\frac{\partial L(P)}{\partial fc_0} = \frac{\partial L(P)}{\partial P_0} \cdot \frac{\partial P_0}{\partial fc_0} + \frac{\partial L(P)}{\partial P_1} \cdot \frac{\partial P_1}{\partial fc_0} + \frac{\partial L(P)}{\partial P_2} \cdot \frac{\partial P_2}{\partial fc_0} \tag{3-17}$$

### 4.4 対数関数の微分

以降、この数式の右辺を粘り強く計算していくことが求められます。まず、式(3-12)より $L(P) = -(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)$ なので、$L(P)$ は対数関数と言えます。よって、これを $P_0$, $P_1$, $P_2$ で微分することは対数関数を微分する操作となります。

ここで重要な公式として、$f(x) = \log x$ について、対数関数の微分の公式より次に示す式が成り立ちます。ただし、真数条件より $x > 0$ です。

$$\frac{d}{dx} f(x) = \frac{d}{dx} \log x = \frac{1}{x}$$

すると、式(3-17)の右辺第1項の左側 $\frac{\partial L(P)}{\partial P_0}$ の計算結果は式(3-18)となります。$P_0$ に関係ない $P_1$, $P_2$ の項は微分操作で消えていることに着目してください。

$$\frac{\partial L(P)}{\partial P_0} = \frac{\partial}{\partial P_0}\{-(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)\} = -\frac{t_0}{P_0} \tag{3-18}$$

In [ ]:
# 対数関数の微分の確認

print("=== 対数関数の微分 ===")
print()
print("公式: d/dx log(x) = 1/x")
print()
print("L(P) = -(t_0 log P_0 + t_1 log P_1 + t_2 log P_2)")
print()
print("∂L(P)/∂P_0 = ∂/∂P_0 {-(t_0 log P_0 + t_1 log P_1 + t_2 log P_2)}")
print("           = -t_0 * (1/P_0) + 0 + 0")
print("           = -t_0/P_0")
print()
print("同様に:")
print("∂L(P)/∂P_1 = -t_1/P_1")
print("∂L(P)/∂P_2 = -t_2/P_2")

In [ ]:
# 対数関数の微分を数値的に確認

def log_func(x):
    return np.log(x)

def log_derivative_analytical(x):
    return 1/x

fig, ax = plt.subplots(figsize=(10, 6))

x = np.linspace(0.1, 5, 100)

ax.plot(x, log_func(x), 'b-', linewidth=2, label=r'$f(x) = \log x$')
ax.plot(x, log_derivative_analytical(x), 'r--', linewidth=2, label=r"$f'(x) = 1/x$")
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.axvline(x=1, color='gray', linestyle=':', alpha=0.5)
ax.axhline(y=1, color='gray', linestyle=':', alpha=0.5)

# x=1での値を示す
ax.scatter([1], [0], color='blue', s=100, zorder=5)
ax.scatter([1], [1], color='red', s=100, zorder=5)
ax.annotate(r'$\log(1) = 0$', xy=(1, 0), xytext=(1.5, 0.5), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='blue'))
ax.annotate(r"$(\log x)' |_{x=1} = 1$", xy=(1, 1), xytext=(2, 1.5), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title(r'対数関数 $\log x$ とその導関数 $1/x$', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 5)
ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()

## 5. 完全な実装：連鎖律による勾配計算

以上の数学的導出をPythonで実装し、数値的に確認してみましょう。

In [ ]:
def softmax(x):
    """ソフトマックス関数"""
    x_shifted = x - np.max(x)  # オーバーフロー対策
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x)

def cross_entropy_loss(P, t):
    """クロスエントロピー損失関数
    L(P) = -Σ t_k log P_k
    """
    return -np.sum(t * np.log(P + 1e-10))  # 数値安定性のため小さな値を追加

class SimpleFullyConnectedLayer:
    """シンプルな全結合層（教科書の例に対応）"""
    
    def __init__(self):
        # 2つの入力 (z1, z2) から 3つの出力 (fc0, fc1, fc2) への重み
        # w[i,j] = z_i から fc_j への重み
        self.w = np.array([
            [0.5, 0.3, 0.2],   # z1 から fc0, fc1, fc2 への重み
            [0.4, 0.5, 0.1],   # z2 から fc0, fc1, fc2 への重み
        ])
    
    def forward(self, z):
        """順伝播
        fc_j = Σ_i w[i,j] * z_i
        """
        self.z = z  # 後で使うため保存
        self.fc = z @ self.w  # [fc0, fc1, fc2]
        return self.fc
    
    def get_partial_fc_partial_w(self, i, j):
        """∂fc_j/∂w_{i,j} を計算
        
        fc_j = w_{1,j} * z_1 + w_{2,j} * z_2
        ∂fc_j/∂w_{i,j} = z_i
        """
        return self.z[i]

# 例を実行
print("=== 全結合層と連鎖律の実装 ===")
print()

# 入力値（プーリング層からの出力）
z = np.array([1.5, 2.0])  # z1=1.5, z2=2.0
print(f"入力 z = {z}  (z1={z[0]}, z2={z[1]})")

# 正解ラベル（one-hot encoding）
t = np.array([1, 0, 0])  # クラス0が正解
print(f"正解ラベル t = {t}  (クラス0が正解)")
print()

# 全結合層の順伝播
fc_layer = SimpleFullyConnectedLayer()
fc = fc_layer.forward(z)
print(f"全結合層の重み:")
print(f"  w[1,0]^fc = {fc_layer.w[0,0]}  (z1 → fc0)")
print(f"  w[2,0]^fc = {fc_layer.w[1,0]}  (z2 → fc0)")
print()
print(f"全結合層の出力:")
print(f"  fc0 = w[1,0]*z1 + w[2,0]*z2 = {fc_layer.w[0,0]}*{z[0]} + {fc_layer.w[1,0]}*{z[1]} = {fc[0]}")
print(f"  fc1 = {fc[1]}")
print(f"  fc2 = {fc[2]}")
print()

# ソフトマックス
P = softmax(fc)
print(f"ソフトマックス出力 P = {P}")
print(f"  P0 = {P[0]:.4f}")
print(f"  P1 = {P[1]:.4f}")
print(f"  P2 = {P[2]:.4f}")
print()

# 損失関数
L = cross_entropy_loss(P, t)
print(f"損失関数 L(P) = {L:.4f}")

In [ ]:
# 連鎖律による勾配計算

print("=== 連鎖律による ∂L/∂w_{1,0}^fc の計算 ===")
print()
print("式(3-15): ∂L(P)/∂w_{1,0}^fc = ∂L(P)/∂fc_0 * ∂fc_0/∂w_{1,0}^fc")
print()

# ステップ1: ∂fc_0/∂w_{1,0}^fc = z_1
dfc0_dw10 = fc_layer.get_partial_fc_partial_w(0, 0)  # i=0 (z1), j=0 (fc0)
print(f"ステップ1: ∂fc_0/∂w_{{1,0}}^fc = z_1 = {dfc0_dw10}")
print()

# ステップ2: ∂L(P)/∂fc_0 を計算
# 式(3-17): ∂L(P)/∂fc_0 = Σ_k (∂L/∂P_k * ∂P_k/∂fc_0)
print("ステップ2: ∂L(P)/∂fc_0 の計算")
print()

# ∂L/∂P_k = -t_k/P_k
dL_dP = -t / (P + 1e-10)
print(f"  ∂L/∂P_0 = -t_0/P_0 = -{t[0]}/{P[0]:.4f} = {dL_dP[0]:.4f}")
print(f"  ∂L/∂P_1 = -t_1/P_1 = -{t[1]}/{P[1]:.4f} = {dL_dP[1]:.4f}")
print(f"  ∂L/∂P_2 = -t_2/P_2 = -{t[2]}/{P[2]:.4f} = {dL_dP[2]:.4f}")
print()

# ソフトマックスの偏微分
# ∂P_i/∂fc_j = P_i(δ_ij - P_j) where δ_ij is Kronecker delta
# ∂P_0/∂fc_0 = P_0(1 - P_0)
# ∂P_1/∂fc_0 = P_1(0 - P_0) = -P_1*P_0
# ∂P_2/∂fc_0 = P_2(0 - P_0) = -P_2*P_0

dP0_dfc0 = P[0] * (1 - P[0])
dP1_dfc0 = -P[1] * P[0]
dP2_dfc0 = -P[2] * P[0]

print("  ソフトマックスの偏微分（∂P_k/∂fc_0）:")
print(f"    ∂P_0/∂fc_0 = P_0(1-P_0) = {P[0]:.4f}*(1-{P[0]:.4f}) = {dP0_dfc0:.4f}")
print(f"    ∂P_1/∂fc_0 = -P_1*P_0 = -{P[1]:.4f}*{P[0]:.4f} = {dP1_dfc0:.4f}")
print(f"    ∂P_2/∂fc_0 = -P_2*P_0 = -{P[2]:.4f}*{P[0]:.4f} = {dP2_dfc0:.4f}")
print()

# ∂L/∂fc_0 = Σ_k (∂L/∂P_k * ∂P_k/∂fc_0)
dL_dfc0 = dL_dP[0] * dP0_dfc0 + dL_dP[1] * dP1_dfc0 + dL_dP[2] * dP2_dfc0
print(f"  ∂L/∂fc_0 = Σ_k (∂L/∂P_k * ∂P_k/∂fc_0)")
print(f"           = ({dL_dP[0]:.4f})*({dP0_dfc0:.4f}) + ({dL_dP[1]:.4f})*({dP1_dfc0:.4f}) + ({dL_dP[2]:.4f})*({dP2_dfc0:.4f})")
print(f"           = {dL_dfc0:.4f}")
print()

# 簡略化: クロスエントロピー + ソフトマックスの場合
# ∂L/∂fc_k = P_k - t_k
dL_dfc0_simple = P[0] - t[0]
print(f"  ※ クロスエントロピー + ソフトマックスの場合の簡略式:")
print(f"     ∂L/∂fc_0 = P_0 - t_0 = {P[0]:.4f} - {t[0]} = {dL_dfc0_simple:.4f}")
print()

# 最終結果
dL_dw10 = dL_dfc0 * dfc0_dw10
print("="*50)
print(f"最終結果: ∂L/∂w_{{1,0}}^fc = ∂L/∂fc_0 * ∂fc_0/∂w_{{1,0}}^fc")
print(f"         = {dL_dfc0:.4f} * {dfc0_dw10}")
print(f"         = {dL_dw10:.4f}")
print()
print(f"簡略式を使った場合:")
dL_dw10_simple = (P[0] - t[0]) * z[0]
print(f"  ∂L/∂w_{{1,0}}^fc = (P_0 - t_0) * z_1 = ({P[0]:.4f} - {t[0]}) * {z[0]} = {dL_dw10_simple:.4f}")

In [ ]:
# 数値微分との比較で検証

def compute_loss(w10, w, z, t):
    """w_{1,0}^fc を変えた時の損失を計算"""
    w_copy = w.copy()
    w_copy[0, 0] = w10
    fc = z @ w_copy
    P = softmax(fc)
    return cross_entropy_loss(P, t)

# 数値微分
h = 1e-5
w10_current = fc_layer.w[0, 0]
numerical_grad = (compute_loss(w10_current + h, fc_layer.w, z, t) - 
                  compute_loss(w10_current - h, fc_layer.w, z, t)) / (2 * h)

print("=== 数値微分による検証 ===")
print()
print(f"解析的な勾配: ∂L/∂w_{{1,0}}^fc = {dL_dw10:.6f}")
print(f"数値微分:    ∂L/∂w_{{1,0}}^fc ≈ {numerical_grad:.6f}")
print(f"差分: {abs(dL_dw10 - numerical_grad):.10f}")
print()
print("→ 解析解と数値解がほぼ一致することを確認！")

In [ ]:
# 重みパラメータの更新と学習の可視化

def train_step(w, z, t, learning_rate=0.1):
    """1ステップの学習"""
    # 順伝播
    fc = z @ w
    P = softmax(fc)
    L = cross_entropy_loss(P, t)
    
    # 勾配計算（クロスエントロピー + ソフトマックスの簡略式）
    # ∂L/∂fc_k = P_k - t_k
    dL_dfc = P - t
    
    # ∂L/∂w_{i,j} = ∂L/∂fc_j * z_i
    grad_w = np.outer(z, dL_dfc)  # [2, 3] の勾配行列
    
    # 重み更新
    w_new = w - learning_rate * grad_w
    
    return w_new, L, P

# 学習の実行
np.random.seed(42)
w = np.random.randn(2, 3) * 0.5  # 初期重み
z = np.array([1.5, 2.0])
t = np.array([1, 0, 0])  # クラス0が正解

losses = []
predictions = []

print("=== 勾配降下法による学習 ===")
print()

for epoch in range(50):
    w, L, P = train_step(w, z, t, learning_rate=0.5)
    losses.append(L)
    predictions.append(P.copy())
    
    if epoch < 5 or epoch % 10 == 9:
        print(f"Epoch {epoch+1:2d}: 損失 = {L:.4f}, P = [{P[0]:.3f}, {P[1]:.3f}, {P[2]:.3f}], 予測クラス = {np.argmax(P)}")

print()
print(f"最終結果: P_0 = {predictions[-1][0]:.4f} (正解クラスの確率)")

In [ ]:
# 学習過程の可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 損失関数の推移
ax = axes[0]
ax.plot(losses, 'b-', linewidth=2)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('損失', fontsize=12)
ax.set_title('損失関数の推移', fontsize=14)
ax.grid(True, alpha=0.3)

# 予測確率の推移
ax = axes[1]
predictions_array = np.array(predictions)
ax.plot(predictions_array[:, 0], 'r-', linewidth=2, label='$P_0$ (正解クラス)')
ax.plot(predictions_array[:, 1], 'g--', linewidth=2, label='$P_1$')
ax.plot(predictions_array[:, 2], 'b:', linewidth=2, label='$P_2$')
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('確率', fontsize=12)
ax.set_title('予測確率の推移', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

print("学習により、正解クラス(クラス0)の確率P_0が1に近づいていることがわかります。")

## まとめ

### 合成関数と連鎖律

$$\{f(g(x))\}' = f'(g(x)) \cdot g'(x)$$

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

### ニューラルネットワークにおける入れ子構造

$$L(P) \leftarrow P_k = \text{softmax}(fc_k) \leftarrow fc_k = \sum_i w_{i,k}^{fc} z_i$$

### 偏微分による重みパラメータの勾配計算

$$\frac{\partial L(P)}{\partial w_{1,0}^{fc}} = \frac{\partial L(P)}{\partial fc_0} \cdot \frac{\partial fc_0}{\partial w_{1,0}^{fc}} = (P_0 - t_0) \cdot z_1$$

### 重みパラメータの更新式

$$w_{1,0}^{fc(n)} = w_{1,0}^{fc(n-1)} - \eta \cdot \frac{\partial L(P)}{\partial w_{1,0}^{fc(n-1)}}$$

### 次のステップ

次回は、この連鎖律を畳み込み層にまで遡って適用する**誤差逆伝播法（バックプロパゲーション）**について詳しく学びます。